# Capítulo 4 - Implementando um modelo GPT do zero para gerar texto

Notebook do Capítulo 4 do livro *Build a Large Language Model (From
Scratch)*, de Sebastian Raschka, seguindo a ordem das seções do notebook
principal do autor (`ch04/01_main-chapter-code/ch04.ipynb`).

Diferença em relação ao livro: as peças reutilizáveis (LayerNorm, GELU,
FeedForward, TransformerBlock, GPTModel e geração) ficam em `src/model/` e
são **importadas** aqui. O `GPTModel` reaproveita o `GPTInputEmbeddings`
da Sprint 2 e a `MultiHeadAttention` da Sprint 3. Só o material didático
do livro (o esqueleto `DummyGPTModel` e a rede de exemplo da seção 4.4) é
definido no próprio notebook.

In [ ]:
import sys
from pathlib import Path

# O notebook roda a partir de sprints/ch04/notebooks/: a raiz do projeto
# fica tres niveis acima.
PROJECT_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(PROJECT_ROOT))

from importlib.metadata import version

import tiktoken
import torch
import torch.nn as nn

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

: 

## 4.1 Codificando a arquitetura de um LLM

- O GPT é um modelo **só-decodificador**: gera um token por vez usando a
  parte *decoder* do Transformer original.
- LLMs são grandes pelo número de parâmetros, não pelo tamanho do código:
  o mesmo bloco se repete várias vezes.
- Implementamos a arquitetura do **GPT-2 small (124M)**. A configuração
  fica em `src/model/config.py`, junto com a configuração reduzida do
  projeto (`GPT_CONFIG_MINI`), usada nos experimentos.

In [2]:
from src.model.config import GPT_CONFIG_124M, GPT_CONFIG_MINI

for chave, valor in GPT_CONFIG_124M.items():
    print(f"{chave:<15} {valor}")

vocab_size      50257
context_length  1024
emb_dim         768
n_heads         12
n_layers        12
drop_rate       0.1
qkv_bias        False


O livro começa **de cima para baixo**: um esqueleto do modelo inteiro em
que o Transformer Block e a LayerNorm são *placeholders* (só devolvem a
entrada). Assim o fluxo completo já roda e dá para conferir os shapes. As
próximas seções substituem cada placeholder pela peça real. O esqueleto é
material de estudo e fica só neste notebook.

In [3]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Placeholder do Transformer Block (secao 4.5)
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # Placeholder da LayerNorm (secao 4.2)
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)                                        # (batch, seq_len, emb_dim)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))   # (seq_len, emb_dim)
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)                                                # (batch, seq_len, vocab_size)
        return logits


class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

    def forward(self, x):
        return x  # nao faz nada: devolve a entrada


class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

    def forward(self, x):
        return x  # nao faz nada: devolve a entrada

Um lote com duas frases de 4 tokens cada, tokenizadas com o BPE do GPT-2
(o mesmo da Sprint 2):

In [4]:
tokenizer = tiktoken.get_encoding("gpt2")

batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


In [5]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

Output shape: torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6755, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


Saída `(2, 4, 50257)`: para cada uma das 4 posições de cada frase, uma
pontuação (**logit**) para cada token do vocabulário ser o **próximo**.
Os valores podem variar entre sistemas; o que importa é o shape.